# Car AI Project - Preprocessing Linear Regression

## 1. Libraries

In [0]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import KBinsDiscretizer

import joblib

In [0]:
# Import project configuration
import sys
import os

# Add parent directory to path to import config
sys.path.append("..")
from config import *

print("PROJECT CONFIGURATION LOADED")
print(f"\nBASE_PATH: {BASE_PATH}")
print(f"\nData Paths:")
print(f"   - RAW_DATA_PATH: {RAW_DATA_PATH}")
print(f"   - FEATURES_PATH: {FEATURES_PATH}")
print(f"   - PROCESSED_DATA_PATH: {PROCESSED_DATA_PATH}")
print(f"   - TRAIN_TEST_PATH: {TRAIN_TEST_PATH}")
print(f"   - METRICS_PATH: {METRICS_PATH}")
print(f"\nModel Path:")
print(f"   - MODEL_PATH: {MODEL_PATH}")
print(f"\nUnity Catalog:")
print(f"   - SOURCE_CSV_FILE: {SOURCE_CSV_FILE}")
print(f"   - RAW_CARS_TABLE: {RAW_CARS_TABLE}")
print(f"   - CLEANED_CARS_TABLE: {CLEANED_CARS_TABLE}")

PROJECT CONFIGURATION LOADED

BASE_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject

Data Paths:
   - RAW_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/raw
   - FEATURES_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/features
   - PROCESSED_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/processed
   - TRAIN_TEST_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/train_test
   - METRICS_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/metrics

Model Path:
   - MODEL_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/models

Unity Catalog:
   - SOURCE_CSV_FILE: /Volumes/workspace/caraiproject/caraiproject/Cars_Datasets_2025.csv
   - RAW_CARS_TABLE: workspace.caraiproject.raw_cars_data_gathered
   - CLEANED_CARS_TABLE: workspace.caraiproject.cleaned_cars_data


## 2. Load the LR Dataset

### 2.1 Import Delta Table in a Pandas Dataframe

In [0]:
# Import unified engineered dataset from Delta Table
# This is the SAME table used by 05_2_Preprocessing_DecisionTree
# Ensures fair comparison and reproducible train/test splits

source_table = "workspace.caraiproject.engineered_cars_data"

# Read Delta Table using Spark (required for Delta format)
spark_df = spark.table(source_table)
row_count_spark = spark_df.count()

# Convert to pandas DataFrame for data manipulation
df = spark_df.toPandas()

# Display dataset information
print("UNIFIED ENGINEERED DATASET")
print(f"\nShape: {df.shape}")

print(f"\nColumns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:30s} ({df[col].dtype})")

display(df.head())

print(f"\nDataset ready for Linear Regression Preprocessing")

UNIFIED ENGINEERED DATASET

Shape: (1209, 32)

Columns (32):
   1. make                           (object)
   2. model                          (object)
   3. engine_displacement_cc         (float64)
   4. battery_capacity_kwh           (float64)
   5. acc_0_100_min                  (float64)
   6. acc_0_100_max                  (float64)
   7. acc_0_100_mean                 (float64)
   8. acc_0_100_is_range             (int64)
   9. acc_0_100_is_instant           (int64)
  10. is_commercial                  (int64)
  11. horsepower_min                 (float64)
  12. horsepower_max                 (float64)
  13. horsepower_mean                (float64)
  14. horsepower_is_range            (int64)
  15. top_speed_kmh                  (float64)
  16. seats                          (float64)
  17. torque_nm                      (int64)
  18. price_usd                      (float64)
  19. is_ev                          (int64)
  20. is_hybrid                      (int64)
  21. is_phev  

make,model,engine_displacement_cc,battery_capacity_kwh,acc_0_100_min,acc_0_100_max,acc_0_100_mean,acc_0_100_is_range,acc_0_100_is_instant,is_commercial,horsepower_min,horsepower_max,horsepower_mean,horsepower_is_range,top_speed_kmh,seats,torque_nm,price_usd,is_ev,is_hybrid,is_phev,fuel_macro,segment,is_luxury_brand,is_performance_luxury_brand,is_premium_brand,acceleration_class,hp_class,log_price,performance_score,acc_missing_flag,fe_timestamp
Ferrari,SF90 STRADALE,3990.0,null,2.5,2.5,2.5,0,0,0,963.0,963.0,963.0,0,340.0,2.0,800,1100000.0,0,1,1,Hybrid,Supercar,0,1,0,Extreme,Extreme,13.910821646859095,105.2,0,2026-05-11T06:59:11.631Z
Rolls-Royce,PHANTOM,6749.0,null,5.3,5.3,5.3,0,0,0,563.0,563.0,563.0,0,250.0,5.0,900,460000.0,0,0,0,Petrol,Sportscar,1,0,0,Medium,High,13.038983942375959,62.5,0,2026-05-11T06:59:11.631Z
Mercedes-Benz,GT 63 S,3982.0,null,3.2,3.2,3.2,0,0,0,630.0,630.0,630.0,0,250.0,4.0,900,161000.0,0,0,0,Petrol,Supercar,0,0,1,Fast,Extreme,11.989165855127435,71.3,0,2026-05-11T06:59:11.631Z
Audi,AUDI R8 Gt,5204.0,null,3.6,3.6,3.6,0,0,0,602.0,602.0,602.0,0,320.0,2.0,560,253290.0,0,0,0,Petrol,Supercar,0,0,1,Fast,Extreme,12.442294304367605,65.4,0,2026-05-11T06:59:11.631Z
BMW,Mclaren 720s,3994.0,null,2.9,2.9,2.9,0,0,0,710.0,710.0,710.0,0,341.0,2.0,770,499000.0,0,0,0,Petrol,Supercar,0,0,1,Extreme,Extreme,13.120363378739663,79.21,0,2026-05-11T06:59:11.631Z



Dataset ready for Linear Regression Preprocessing


In [0]:
# drop the cleaning_timestamp column and create a new DataFrame
df_lr = df.drop(columns=["fe_timestamp"])

## 3. Preprocessing

**Unified Base + Linear Regression Adaptations**

This preprocessing notebook starts from the **same engineered dataset** used by Decision Trees (05_2).  
All models see the **same features initially**, ensuring fair comparison.

**Linear Regression-Specific Adaptations:**

1. **Remove structural features** (`engine_displacement_cc`, `battery_capacity_kwh`)  
   - These have **structural missingness** tied to powertrain type  
   - Their information is already encoded in `is_ev`, `is_hybrid`, `fuel_macro`  
   - Linear models cannot naturally handle structural nulls without leakage risk

2. **Log-transform target** (`log_price`)  
   - Stabilizes variance  
   - Reduces outlier influence  
   - Improves linearity assumptions

3. **Standard scaling** (applied in pipeline)  
   - Linear models are sensitive to feature scale  
   - Tree-based models do not require this

All other features remain identical to the unified base from 04_Feature_Engineering, including:  
`performance_score` (composite performance indicator)  
`acc_missing_flag` (structural missingness indicator)  
Brand tiers, segments, and all derived features

### 3.1 Split Train/Test

In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1209 entries, 0 to 1208
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   make                         1209 non-null   object        
 1   model                        1209 non-null   object        
 2   engine_displacement_cc       1061 non-null   float64       
 3   battery_capacity_kwh         104 non-null    float64       
 4   acc_0_100_min                1204 non-null   float64       
 5   acc_0_100_max                1204 non-null   float64       
 6   acc_0_100_mean               1204 non-null   float64       
 7   acc_0_100_is_range           1209 non-null   int64         
 8   acc_0_100_is_instant         1209 non-null   int64         
 9   is_commercial                1209 non-null   int64         
 10  horsepower_min               1209 non-null   float64       
 11  horsepower_max               1209 non-null 

In [0]:
# Create price bins for stratified train/test split
# Ensures proportional representation of all price ranges (including supercars) in train and test
# Using price_usd for bins: more interpretable than log_price (reflects actual market segments)
price_bins = pd.qcut(df_lr['price_usd'], q=10, labels=False, duplicates='drop')

print(f"Price bins created: {price_bins.nunique()} bins")
print(f"Samples per bin:")
print(price_bins.value_counts().sort_index())

Price bins created: 10 bins
Samples per bin:
price_usd
0    128
1    141
2    124
3     96
4    117
5    130
6    110
7    121
8    121
9    121
Name: count, dtype: int64


In [0]:
# Linear Regression: Drop structural features with missingness
# These features' information is already encoded in powertrain indicators
X = df_lr.drop(columns=[
    'price_usd',                # Target (raw)
    'log_price',                # Target (log-transformed)
    'model',                     # Identifier, not a feature
    'engine_displacement_cc',   # Structural missingness (ICE only)
    'battery_capacity_kwh'      # Structural missingness (EV only)
])

y = df_lr['log_price']

# Split with stratification to ensure proportional representation of all price ranges
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=3, stratify=price_bins
)

print("✅ Stratified train/test split completed (log_price target)")
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   Features retained: {X_train.shape[1]}")
print(f"   Stratified by: price_usd bins (ensures all price ranges represented)")

✅ Stratified train/test split completed (log_price target)
   X_train: (967, 26)
   X_test: (242, 26)
   Features retained: 26
   Stratified by: price_usd bins (ensures all price ranges represented)


#### 3.1.1 Horsepower Bins only Train

In [0]:
kbins = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')

X_train["hp_bin"] = kbins.fit_transform(X_train[["horsepower_mean"]])
X_test["hp_bin"] = kbins.transform(X_test[["horsepower_mean"]])

In [0]:
X_train["hp_bin"].value_counts()

hp_bin
3.0    252
0.0    242
1.0    241
2.0    232
Name: count, dtype: int64

### 3.2 Pipeline

In [0]:
# Features
numerical_features = [
    "acc_0_100_mean",
    "acc_0_100_min",
    "acc_0_100_max",
    "horsepower_mean",
    "horsepower_min",
    "horsepower_max",
    "top_speed_kmh",
    "torque_nm",
    "acc_0_100_is_range",
    "acc_0_100_is_instant",
    "horsepower_is_range",
    "acc_missing_flag",
    "performance_score",
    "seats",
    "hp_bin"
]
categorical_features = [
    "make",
    "fuel_macro",
    "acceleration_class",
    "hp_class",
    "segment",
]

binary_features = [
    "is_ev",
    "is_hybrid",
    "is_phev",
    "is_commercial",
    "is_luxury_brand",
    "is_premium_brand",
    "is_performance_luxury_brand"
]



# Pipelines
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

seat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, [f for f in numerical_features if f != 'seats']),
        ('seat', seat_pipeline, ['seats']),
        ('cat', categorical_pipeline, categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)

# Final pipeline
model_lr = Pipeline(steps=[
    ('preprocess', preprocess),
    ('regressor', LinearRegression())
])

**Note**:All imputations are performed after the train-test split to avoid data leakage.

Numerical features are imputed using the median, while the `seats` feature is treated separately using the most frequent value due to its discrete nature.

The final preprocessing pipeline applies:
- Standard scaling to numerical features,
- One-hot encoding to categorical features,
- Linear Regression as the predictive model.

This ensures a clean, reproducible and leakage-safe modeling workflow.

### 3.3 Save Objects

In [0]:
# Save train/test splits
joblib.dump(X_train, os.path.join(TRAIN_TEST_PATH, "X_train_lr.pkl"))
joblib.dump(X_test, os.path.join(TRAIN_TEST_PATH, "X_test_lr.pkl"))
joblib.dump(y_train, os.path.join(TRAIN_TEST_PATH, "y_train_lr.pkl"))
joblib.dump(y_test, os.path.join(TRAIN_TEST_PATH, "y_test_lr.pkl"))

# Save feature lists
joblib.dump(numerical_features, os.path.join(FEATURES_PATH, "numerical_features_lr.pkl"))
joblib.dump(categorical_features, os.path.join(FEATURES_PATH, "categorical_features_lr.pkl"))
joblib.dump(binary_features, os.path.join(FEATURES_PATH, "binary_features_lr.pkl"))

# Save pipeline
joblib.dump(model_lr, os.path.join(TRAIN_TEST_PATH, "pipeline_lr.pkl"))

['/Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/train_test/pipeline_lr.pkl']

#### Pipeline for Target Variable without Transformation

In [0]:
# Pipeline for price_usd target (no log transformation)
# Same feature exclusions as log_price pipeline

X_usd = df_lr.drop(columns=[
    'price_usd',                # Target
    'log_price',                # Alternative target
    'model',                     # Identifier
    'engine_displacement_cc',   # Structural missingness
    'battery_capacity_kwh',     # Structural missingness
    'torque_nm',                 
    'performance_score'          # Composite of other features
])

y_usd = df_lr['price_usd']

# Split using THE SAME stratification bins as log_price
# This ensures log and USD models have IDENTICAL train/test row splits
X_train_usd, X_test_usd, y_train_usd, y_test_usd = train_test_split(
    X_usd, y_usd, test_size=0.2, random_state=3, stratify=price_bins
)

print(f"✅ Stratified train/test split completed (price_usd target)")
print(f"   X_train_usd: {X_train_usd.shape}")
print(f"   X_test_usd: {X_test_usd.shape}")
print(f"   Same rows as log_price split (only target differs)")

# Features (same structure as log_price, adjusted for dropped columns)
numerical_features_usd = [
    "acc_0_100_mean",
    "acc_0_100_min",
    "acc_0_100_max",
    "horsepower_mean",
    "horsepower_min",
    "horsepower_max",
    "top_speed_kmh",
    "acc_0_100_is_range",
    "acc_0_100_is_instant",
    "horsepower_is_range",
    "acc_missing_flag",
    "seats"
]

categorical_features_usd = [
    "make",
    "fuel_macro",
    "acceleration_class",
    "hp_class",
    "segment",
]

binary_features_usd = [
    "is_ev",
    "is_hybrid",
    "is_phev",
    "is_commercial",
    "is_luxury_brand",
    "is_performance_luxury_brand",
    "is_premium_brand"
]

# Build pipeline (same structure as log_price)
numerical_pipeline_usd = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline_usd = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_usd = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline_usd, numerical_features_usd),
        ('cat', categorical_pipeline_usd, categorical_features_usd),
        ('bin', 'passthrough', binary_features_usd)
    ])

model_lr_usd = Pipeline(steps=[
    ('preprocessor', preprocessor_usd),
    ('regressor', LinearRegression())
])

print("✅ USD pipeline created")

✅ Stratified train/test split completed (price_usd target)
   X_train_usd: (967, 24)
   X_test_usd: (242, 24)
   Same rows as log_price split (only target differs)
✅ USD pipeline created


In [0]:
# Save USD train/test splits
joblib.dump(X_train_usd, os.path.join(TRAIN_TEST_PATH, "X_train_lr_usd.pkl"))
joblib.dump(X_test_usd, os.path.join(TRAIN_TEST_PATH, "X_test_lr_usd.pkl"))
joblib.dump(y_train_usd, os.path.join(TRAIN_TEST_PATH, "y_train_lr_usd.pkl"))
joblib.dump(y_test_usd, os.path.join(TRAIN_TEST_PATH, "y_test_lr_usd.pkl"))

# Save USD feature lists
joblib.dump(numerical_features_usd, os.path.join(FEATURES_PATH, "numerical_features_lr_usd.pkl"))
joblib.dump(categorical_features_usd, os.path.join(FEATURES_PATH, "categorical_features_lr_usd.pkl"))
joblib.dump(binary_features_usd, os.path.join(FEATURES_PATH, "bin_features_lr_usd.pkl"))

# Save USD pipeline
joblib.dump(model_lr_usd, os.path.join(TRAIN_TEST_PATH, "pipeline_lr_usd.pkl"))

['/Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/train_test/pipeline_lr_usd.pkl']